# '03.-Validaciones Contables Marine' - PROGRAM 03

##  Overview
In this program, the processed database, and the account files are loaded.
First, a validation of OSLR is made.
Second, a payments validation is made.
Finally, both validations are exported in a single Excel File

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import(Processed Database and Account information). 
4. Data Process.
5. OSLR Validation.
6. Payments Validation.
7. Final Export.

##  Output
- Excel file with OSLR and payments validation. ({AñoMes}_Validaciones_Contables_Marine.xlsx)

## 1. Library Import

In [79]:
# =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import timeit
import sqlite3

start_time = timeit.default_timer() # Timer

## 1.5 Importación de configuración centralizada

In [80]:
# =============================================================================
# Importación centralizada de configuración
# =============================================================================
import sys
sys.path.insert(0, r'C:/Users/IKAL14/Documents/Integral/Marine/Diccionarios')
from config_marine import *


## 2. Path Definition and Macrovariables

In [81]:
# =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================
print('===============================================================================================')
print('Inicio del proceso')
start_time = timeit.default_timer() # Timer
# ================= Edit variables month================================
AñoMes = 202604 # Variable that represents the month of processing
AñoMes_ant = 202512  # Vaariable that represents previous month of processing, default = 1 
# Edit variables yearly
AñoMes_piv = 202412 # December from previous year

# Convert the variables in datetime 
date_start_AñoMes = pd.to_datetime(str(AñoMes), format='%Y%m') + pd.offsets.MonthBegin(0)
date_start_AñoMes_piv = pd.to_datetime(str(AñoMes_piv), format='%Y%m') + pd.offsets.MonthBegin(1)
date_start_pivot = date_start_AñoMes_piv + pd.offsets.MonthBegin(0) # This is an Aux Variable and it should be used when we are proccesing more than 1 month

#print(f'Fecha de inicio para considerar en la suma: {date_start_pivot}')

# =================Routes definition=========================================

print('===============================================================================================')
#Define your project directory path as a variable
directorio_proyecto = DIRECTORIO_PROYECTO  # Desde config_marine.py

#Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

#Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

#Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados/{AñoMes}" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

#Incidences files path
ruta_incidencias = rf"{directorio_proyecto}/Incidencias/{AñoMes}" 
print(f"Ruta de Incidencias: {ruta_incidencias}")

#Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")

#Catalog files path
route_catalog = rf'{directorio_proyecto}/Catalogos'
print(f'Ruta de catalogos: {route_catalog}')

# Actuarial database path
route_actuarial = RUTA_INSUMOS  # Desde config_marine.py
print(f'Ruta de base actuarial: {route_actuarial}')

# Payments information
route_account = RUTA_CONTABILIDAD  # Desde config_marine.py


Inicio del proceso
Directorio actual de trabajo: C:\Users\IKAL14\Documents\Integral\Marine
Ruta de archivos procesados: C:/Users/IKAL14/Documents/Integral/Marine/Procesados/202604
Ruta de Incidencias: C:/Users/IKAL14/Documents/Integral/Marine/Incidencias/202604
Ruta de Validaciones: C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202604
Ruta de catalogos: C:/Users/IKAL14/Documents/Integral/Marine/Catalogos
Ruta de base actuarial: C:/Users/IKAL14/Documents/Integral/Insumos


In [82]:
# ======== Import of final database ======== #
df_update_db = pd.read_pickle(f'{ruta_procesados}/Final/{AñoMes}_Siniestros_Marine.pkl')
#df_update_db = pd.read_excel(f'{ruta_procesados}/pruebas/Marine_{AñoMes}.xlsx')
#df_update_db.to_pickle(f'{ruta_procesados}/pruebas/Marine_{AñoMes}.pkl')

## 3. Data Import

In [83]:
# =============================================================================
# Section 3: Data import
# =============================================================================

# ======== Import of final database ======== #
#df_update_db = pd.read_pickle(f'{ruta_procesados}/pruebas/Marine_{AñoMes}.pkl') #Merged.xlsx
# String format
for col in ['INWARD POLICY N°','LoB-Inward']:
    df_update_db[col] = df_update_db[col].apply(lambda x: str(x).strip() if pd.notna(x) else x)


# ======== PAYMENTS information (December last year) ======== #
# Uncomment if there´s an update #
#payments_pivote = pd.read_excel(fr'{route_account}/{AñoMes_piv}/Payments_{AñoMes_piv}.xlsx')
#payments_pivote.to_pickle(fr'{route_account}/Payments_{AñoMes_piv}.pkl')
# ============================================================== #
# This is the picture of the PAYMENTS to december of last year
df_payments_pivote = pd.read_excel(fr'{route_account}/{AñoMes}/Payments_{AñoMes}.xlsx')
# String format
#df_payments_pivote['Póliza'] = df_payments_pivote['Póliza'].apply(lambda x: str(x).strip() if pd.notna(x) else x)
# Data format
#df_payments_pivote.loc[df_payments_pivote['Póliza']=='90600-323484','Póliza'] = '90600 323484'
#df_payments_pivote.loc[df_payments_pivote['Póliza']=='90600-328256','Póliza'] = '90600 328256'


# ======= Account information(OSLR MONTH) ======= #
df_oslr_conta_original = pd.read_excel(f'{route_account}/{AñoMes}/OSLR {AñoMes}.xlsx')
# Rename of policies
df_oslr_conta_original.loc[df_oslr_conta_original['Policy Number']== '90600 00320575' , 'Policy Number'] = '90600 320575'
# String format
for col in ['Policy Number', 'Cover']:
    df_oslr_conta_original[col] = df_oslr_conta_original[col].apply(lambda x: str(x).strip() if pd.notna(x) else x)


# ======== PAYMENTS information(File of the current month) ======== #
# Initialize a dictionary with the sheets of information
dict_sheets = pd.read_excel(fr'{route_account}/{AñoMes}/Payments_{AñoMes}.xlsx', sheet_name= None)
# Split the dictionary in dataframes
df_AE = dict_sheets['AE']  # Expenses Paid
df_CL = dict_sheets['CL']  # Claims Paid
df_VA = dict_sheets['VA']  # Valuation Expenses

# Cleaning
del dict_sheets

## 4. Data Process

In [84]:
# =============================================================================
# Section 4: Data Process
# =============================================================================

# ======= Final Database ======= #
# Creation of numeric validation file
df_oslr_base = df_update_db.groupby(['INWARD POLICY N°', 'LoB-Inward']).agg(
    OSLR_Inward =('OSLR Inward', 'sum')
).reset_index()


# ======= Account information (OSLR) ======= #

df_oslr_conta = df_oslr_conta_original.groupby(['Policy Number','Cover']).agg(
    OSLR_Conta = (f'Net Reserve', 'sum')
).reset_index()

# Validation of total amount
total_oslr_conta = df_oslr_conta['OSLR_Conta'].sum()
print('Tenemos que la suma total en el archivo de contabilidad es:')
print(f'${total_oslr_conta:,.2f}')

# ======== PAYMENTS information ======== #

# Initialize a list of dataframes, since they have the same structure
list_df_payments = [df_AE,df_CL,df_VA]

# ====== Format Variables ====== #

# === Date Variables === #
# Select the columns to be formated as date
cols_date = ['Cover period from', 'Cover period to','Claim Date','Claim Report', 'Date of payment','Payment request date']
# Iterate over the dataframes and columns
for i in range(len(list_df_payments)):
    for col in cols_date:
        list_df_payments[i][col] = pd.to_datetime(list_df_payments[i][col], dayfirst= True , errors= 'coerce') 

# === String Variables === #
cols_string = ['Claims Reference','Lob-Inward','Policy N°']
# Delete all spaces (leading, trailing, and in between) in CLAIM NUMBER
for i in range(len(list_df_payments)):
    for col in cols_string: # Delete all the spaces at the beggining, end or in the middle
        list_df_payments[i][col] = list_df_payments[i][col].apply(lambda x: str(x).strip().replace(' ', '') if not pd.isna(x) else x)

# ====== Mormalization of Lob's and Policy N¬ ====== #
for i in range(len(list_df_payments)):
    # Rename LoB inward from 'P&I' to 'PANDI'
    list_df_payments[i].loc[list_df_payments[i]['Lob-Inward']=='P&I','Lob-Inward'] = 'PANDI' 
    # Rename LoB inward from 'P&I' to 'PANDI'
    list_df_payments[i].loc[list_df_payments[i]['Lob-Inward']=='CASCO','Lob-Inward'] = 'CASCO Y MAQUINARIA' 
    # Rename 'Policy N°' from 'E01-2-60-10' to 'E01-2-60-000000010_0000-0-1'
    list_df_payments[i].loc[list_df_payments[i]['Policy N°'] == 'E01-2-60-10','Policy N°'] = 'E01-2-60-000000010_0000-0-1'
    # Rename 'Policy N°' from 'E01-2-60-3' to 'E01-2-60-000000003_0000-0-1'
    list_df_payments[i].loc[list_df_payments[i]['Policy N°'] == 'E01-2-60-3','Policy N°'] = 'E01-2-60-000000003_0000-0-1'
    # Rename LoB inward from 'CascoyMaquinaria' to 'CASCO Y MAQUINARIA' 
    list_df_payments[i].loc[list_df_payments[i]['Lob-Inward']=='CascoyMaquinaria','Lob-Inward'] = 'CASCO Y MAQUINARIA' 


# Outstanding information of CL
df_CL_filtered = df_CL[df_CL['Status'] == 'Outstanding'].groupby(['Policy N°','Lob-Inward']).agg(
    OSLR_CL = (f'Amount USD (CORRECT)', 'sum')
).reset_index()
total_conta_outstanding = df_CL_filtered['OSLR_CL'].sum()
print("El OSLR de siniestros que tienen status 'outstanding' en el archivo contable de claims es:")
print(f'${total_conta_outstanding:,.2f}')

Tenemos que la suma total en el archivo de contabilidad es:
$1,714,322.34
El OSLR de siniestros que tienen status 'outstanding' en el archivo contable de claims es:
$1,585,482.78


In [85]:
# ================= Final database information Handling ================= #

# Drop the records with OSLR Inward equal 0
df_oslr_base = df_oslr_base[~(df_oslr_base['OSLR_Inward'] == 0)]

# Optional: Debug
#df_oslr_base.to_excel(f'{ruta_procesados}/pruebas/oslr_base_marine.xlsx', index= False)

In [86]:
# ================= Account information Handling ================= #

# Initialize the list of policies that are considered in Marine Database
list_policies_marine = df_update_db['INWARD POLICY N°'].unique()

# Identify policies not considered as Marine
list_policies_nomarine = list(set(df_oslr_conta[~df_oslr_conta['Policy Number'].isin(list_policies_marine)]['Policy Number'].unique()))

# Optional:Debug
# Create DataFrame and fill mismatched lengths with NaN automatically
#df_list_policies = pd.DataFrame({'Marine': pd.Series(list_policies_marine), 'NoMarine': pd.Series(list_policies_nomarine)})
#df_list_policies.to_excel(f'{ruta_procesados}/policies_comparison.xlsx', index=False)
#print(f'Las pólizas consideradas para Marine son las siguientes: {list_policies_marine}')
#print(f'Las pólizas no consideradas para Marine son: {list_policies_nomarine}')

# Verification of the policies considered in the validation
policies_in_conta = df_oslr_conta['Policy Number'].unique()

if set(list_policies_marine).issubset(policies_in_conta):
    print('Todas las pólizas de la base mensual están consideradas en la base contable.')
else:
    missing_policies = set(list_policies_marine) - set(policies_in_conta)
    for policy in missing_policies:
        print(f'La póliza {policy} no se encuentra en la base contable, revisar.')

# === Data cleaning === #
# String format
# Convert 'Policy Number' and 'Cover' to string safely without losing numeric data

# Slice the account information 
df_oslr_conta_marine = df_oslr_conta[df_oslr_conta['Policy Number'].isin(list_policies_marine)] 
# Drop the records with OSLR Inward equal 0
df_oslr_conta_marine = df_oslr_conta_marine[~(df_oslr_conta_marine['OSLR_Conta'] == 0)]  # Review others
# Rename of the cover
df_oslr_conta_marine.loc[df_oslr_conta_marine['Cover'] == 'Casco', 'LoB-Inward Norm'] = 'CASCO Y MAQUINARIA'
df_oslr_conta_marine.loc[df_oslr_conta_marine['Cover'] == 'Transporte', 'LoB-Inward Norm'] = 'CARGO'
df_oslr_conta_marine.loc[df_oslr_conta_marine['Cover'] == 'Pandi', 'LoB-Inward Norm'] = 'PANDI'
df_oslr_conta_marine.loc[df_oslr_conta_marine['Cover'] == 'Carga', 'LoB-Inward Norm'] = 'CARGO'

# Special
df_oslr_conta_marine.loc[(df_oslr_conta_marine['Cover'] == 'Ferreo') & (df_oslr_conta_marine['Policy Number'] == '3612100000008'),'LoB-Inward Norm'] = 'CARGA POLIETILENO'
df_oslr_conta_marine.loc[(df_oslr_conta_marine['Cover'] == 'Ferreo') & (df_oslr_conta_marine['Policy Number'] == 'E01-2-60-000000010_0000-0-1'),'LoB-Inward Norm'] = 'EQUIPO FERROVIARIO(DAÑO FÍSICO)'

# Optional: Debug
#df_oslr_conta_marine.to_excel(f'{ruta_procesados}/pruebas/oslr_conta_marine.xlsx', index= False)


La póliza BJ2000120100 no se encuentra en la base contable, revisar.
La póliza 3612100000008 no se encuentra en la base contable, revisar.
La póliza 90600 323484 no se encuentra en la base contable, revisar.
La póliza 38417374 no se encuentra en la base contable, revisar.
La póliza 147736177 no se encuentra en la base contable, revisar.
La póliza M9000324 no se encuentra en la base contable, revisar.
La póliza BJ2000120000 no se encuentra en la base contable, revisar.
La póliza 13637426 no se encuentra en la base contable, revisar.
La póliza 90600 328256 no se encuentra en la base contable, revisar.
La póliza 90600 320575 no se encuentra en la base contable, revisar.
La póliza M9000325 no se encuentra en la base contable, revisar.
La póliza 25300 30014476 no se encuentra en la base contable, revisar.
La póliza BJ200001 no se encuentra en la base contable, revisar.
La póliza CJ2000250200 no se encuentra en la base contable, revisar.


In [87]:
# ================= Cumulative CLAIMS PAID information Handling ================= #

#HANDLED IN CELL 7, DELETE
# Normalize the number of policies in between databases
df_CL_filtered.loc[df_CL_filtered['Policy N°'] == 'E01-2-60-10','Policy N°'] = 'E01-2-60-000000010_0000-0-1'

# Normalize the LoB
df_CL_filtered.loc[df_CL_filtered['Lob-Inward'] == 'P&I', 'Lob-Inward'] = 'PANDI'

# Selection of only marine
df_CL_filtered = df_CL_filtered[df_CL_filtered['Policy N°'].isin(list_policies_marine)] 


## 5. OSLR Validation

In [88]:
# # =============================================================================
# Section 5: OSLR Validation
# ===============================================================================

# === Initialize the keys to merge the databases === #
df_oslr_base.columns = df_oslr_base.columns.str.strip()
df_oslr_conta_marine.columns = df_oslr_conta_marine.columns.str.strip()
df_CL_filtered.columns = df_CL_filtered.columns.str.strip()
# == Final database == #
if 'INWARD POLICY N°' in df_oslr_base.columns:
    df_oslr_base['KEY'] = df_oslr_base['INWARD POLICY N°'] + "-" + df_oslr_base['LoB-Inward']
    df_oslr_base.rename(columns={'OSLR_Inward': 'OSLR BASE'}, inplace= True)
    df_oslr_base = df_oslr_base[['KEY','OSLR BASE']]

# == Account information == #
if 'Policy Number' in df_oslr_conta_marine.columns:
    df_oslr_conta_marine['KEY'] = df_oslr_conta_marine['Policy Number'] + "-" + df_oslr_conta_marine['LoB-Inward Norm']
    df_oslr_conta_marine.rename(columns={'OSLR_Conta': 'OSLR CONTA'}, inplace= True)
    df_oslr_conta_marine = df_oslr_conta_marine[['KEY','OSLR CONTA']]

# == CCP == #
if 'Policy N°' in df_CL_filtered.columns:
    df_CL_filtered['KEY'] = df_CL_filtered['Policy N°'] + "-" +  df_CL_filtered['Lob-Inward'] 
    df_CL_filtered.rename(columns={'OSLR_CL': 'OUTSTANDING'}, inplace= True)
    df_CL_filtered = df_CL_filtered[['KEY','OUTSTANDING']]

# Quick Validation before merge #
print('===============================================================')
print('VALIDACION DE CIFRAS ANTES DEL CRUCE')
print('===============================================================')
total_oslr_base_marine = df_oslr_base['OSLR BASE'].sum()
total_oslr_conta_marine = df_oslr_conta_marine['OSLR CONTA'].sum()
diference_conta_marine = total_oslr_base_marine - total_oslr_conta_marine 
total_oslr_outstanding = df_CL_filtered['OUTSTANDING'].sum()

print('El total que tenemos de OSLR en la base final de marine es:')
print(f'${total_oslr_base_marine:,.2f}')

print('El total que tenemos de OSLR en la base contable para marine es:')
print(f'${total_oslr_conta_marine:,.2f}')

print('La diferencia es de:')
print(f'${diference_conta_marine:,.2f}')

print('El total que tenemos de OSLR en outstanding para marine es:')
print(f'${total_oslr_outstanding:,.2f}')
print('===============================================================')

# Debug: encontrar KEYs que difieren solo en CARGA/CARGO
keys_base = set(df_oslr_base['KEY'].unique())
keys_conta = set(df_oslr_conta_marine['KEY'].unique())

# KEYs en base que no matchean en conta
sin_match_base = keys_base - keys_conta
carga_keys = [k for k in sin_match_base if 'CARGA' in k]
print(f"KEYs con CARGA sin match en conta: {len(carga_keys)}")
for k in sorted(carga_keys):
    cargo_version = k.replace('CARGA', 'CARGO')
    existe = '(EXISTE como CARGO en conta)' if cargo_version in keys_conta else ''
    print(f"  {k}  {existe}")

# === Cross between final database and account information === #
df_val_1 = pd.merge(
    df_oslr_base,
    df_oslr_conta_marine, 
    on='KEY',
    how='inner')

# === Cross between df_val_1 and CCP === #
df_val_1 = pd.merge(
    df_val_1,
    df_CL_filtered, 
    on='KEY',
    how='outer')

# Quick Validation after merge #
print('===============================================================')
print('VALIDACION DE CIFRAS DESPUES DEL CRUCE')
print('===============================================================')
total_oslr_base_marine = df_val_1['OSLR BASE'].sum()
total_oslr_conta_marine = df_val_1['OSLR CONTA'].sum()
total_oslr_outstanding = df_val_1['OUTSTANDING'].sum()

print('El total que tenemos de OSLR en la base final de marine es:')
print(f'${total_oslr_base_marine:,.2f}')

print('El total que tenemos de OSLR en la base contable para marine es:')
print(f'${total_oslr_conta_marine:,.2f}')

print('El total que tenemos de OSLR en outstanding para marine es:')
print(f'${total_oslr_outstanding:,.2f}')
print('===============================================================')


# === Final Format === #

# Fill NaN with 0 in specific columns
df_val_1[['OSLR BASE', 'OSLR CONTA', 'OUTSTANDING']] = df_val_1[['OSLR BASE', 'OSLR CONTA', 'OUTSTANDING']].fillna(0)

df_val_1['DIFERENCIAS'] = df_val_1['OSLR BASE'] + df_val_1['OUTSTANDING'] -  df_val_1['OSLR CONTA']  

# Comment for validation
# 1. Initialize the column with a default value (e.g., empty string or None)
df_val_1['COMENTARIOS'] = ''
# 2. Now you can safely use .loc, even if the DataFrame is empty

# Also, using .round() or np.isclose() is safer than .astype(int) for float math!
df_val_1.loc[df_val_1['DIFERENCIAS'].round().astype(int) == 0, 'COMENTARIOS'] = 'OK'

# Final Export
print('===============================================================')
print(f'VALIDACION CONTABLE DE OSLR AL MES {AñoMes} realizada correctamente')
print('===============================================================')

#df_val_1.to_excel(f'{ruta_procesados}/pruebas/validacion_OSLR_{AñoMes}.xlsx', index=False)


VALIDACION DE CIFRAS ANTES DEL CRUCE
El total que tenemos de OSLR en la base final de marine es:
$7,671,689.95
El total que tenemos de OSLR en la base contable para marine es:
$1,714,322.34
La diferencia es de:
$5,957,367.61
El total que tenemos de OSLR en outstanding para marine es:
$0.00
KEYs con CARGA sin match en conta: 0
VALIDACION DE CIFRAS DESPUES DEL CRUCE
El total que tenemos de OSLR en la base final de marine es:
$0.00
El total que tenemos de OSLR en la base contable para marine es:
$0.00
El total que tenemos de OSLR en outstanding para marine es:
$0.00
VALIDACION CONTABLE DE OSLR AL MES 202604 realizada correctamente


## 6. Payments Validation

In [89]:
# # =============================================================================
# Section 6: Payments Validation
# ===============================================================================

# === Initialize the keys to merge the databases === #

# ======= Database Information ======= #
# Creation of numeric validation file
# In this part of the code we pass a dictionary explicitaly to create the columns with the suffix {AñoMes} dinamically
df_payments_base = df_update_db.groupby(['INWARD POLICY N°']).agg(
    **{
        f'Expenses_Paid_{AñoMes}' : ('Cumulative EXPENSES PAID', 'sum'),
        f'Valuation_Expenses_{AñoMes}' : ('Cumulative VALUATION EXPENSES', 'sum'),
        f'Claims_Paid_{AñoMes}' : ('Cumulative CLAIMS PAID', 'sum')
    }
).reset_index()

# Data format
# Rename to normalize the policy number column
df_payments_base.rename(columns = {'INWARD POLICY N°': 'Póliza'}, inplace = True)
# Debug
#df_payments_base.to_excel(f'{ruta_procesados}/pruebas/df_payments_base.xlsx', index= False)


# ======== PAYMENTS information (December last year) ======== #

# Select only marine policies
#df_payments_pivote = df_payments_pivote[df_payments_pivote['Policy N°'].isin(list_policies_marine)]
#print(list_policies_marine)

# Creation of numeric validation file  ##Acumulado al 12/2024
codigos_poliza = [
'13637426','38417374','147736177','3612100000008','25200 30002933',
'25200 30006350','25300 30011610','90600 320575','90600 323484','90600 328256',
'BJ200001','BJ2000120000','BJ2000120100','CJ2000250200','E01-2-60-000000003_0000-0-1',
'E01-2-60-000000010_0000-0-1','M9000324','M9000325','NCGL-070-1000773']
data={
'Expenses_Paid_202412': [
        476156.56, 259721.76, 706841.42, 398520.90, 110720.64, 
        425362.24, 193399.40, 209466.88, 3424.35, 0, 
        0, 0, 114866.15, 25258.75, 517285.42, 
        16162.13, 0, 0, 89731.79
    ],
    'Valuation_Expenses_202412': [
        192145.52, 2545.03, 0, 0, 0, 
        51607.56, 0, 0, 0, 0, 
        0, 0, 0, 1616.90, 299006.02, 
        7850.32, 0, 0, 0
    ],
    'Claims_Paid_202412': [ 
        6575241.94, 3477995.06, 31560354.70, 7394647.53, 5424361.86, 
        14167314.80, 4902819.95, 1773946.42, 366032.23, 49145.73, 
        0, 0, 6742329.03, 226110.07, 5279068.46, 
        3760396.11, 0, 104952.79, 15335485.50
    ]}
data['Póliza'] = codigos_poliza
df = pd.DataFrame(data)
# Reordenar las columnas para que 'Policy ID' sea la primera
df_payments_piv= df[['Póliza', 'Expenses_Paid_202412', 'Valuation_Expenses_202412', 'Claims_Paid_202412']]

#Debug
#df_payments_piv.to_excel(f'{ruta_procesados}/pruebas/df_payments_piv.xlsx', index= False)


# ======== PAYMENTS information (Files of the current month) ======== #
# Select only the columns of interest
list_columns_payments = ['Policy N°','Payment request date', 'Claims Reference', 'Amount USD (CORRECT)','Status' ]


# Filtering the dataframes based on the 'Payment request date' being greater than or equal to 'date_start_pivot'
df_AE_year = df_AE[(df_AE['Payment request date'] >= date_start_pivot) & (df_AE['Policy N°'].isin(list_policies_marine)) ][list_columns_payments]

df_CL_year = df_CL[(df_CL['Payment request date'] >= date_start_pivot) & (df_CL['Policy N°'].isin(list_policies_marine))][list_columns_payments]

df_VA_year = df_VA[(df_VA['Payment request date'] >= date_start_pivot) & (df_VA['Policy N°'].isin(list_policies_marine))][list_columns_payments]

# Dictionary of original dataframes
dict_df_original = {'df_AE_year': df_AE_year, 'df_CL_year':df_CL_year, 'df_VA_year': df_VA_year}
# Dictionay to save grouped dataframes
dict_df_grouped = {}
# Group of the dataframes inside the lopp
for name, df in dict_df_original.items():
    grouped_name = f'{name}_final' # Name of the grouped dataframe
    dict_df_grouped[grouped_name] = df.groupby('Policy N°').agg(
        Monto = ('Amount USD (CORRECT)','sum' )
).reset_index()

# Acces to the grouped dataframes
df_AE_final = dict_df_grouped['df_AE_year_final']
df_CL_final = dict_df_grouped['df_CL_year_final']
df_VA_final = dict_df_grouped['df_VA_year_final']

# ======= DATA CROSS FOR PAYMENTS VALIDATION ======= #
# First merge (AE-VA)
df_payments_month = pd.merge(
    df_AE_final,
    df_VA_final, 
    on = 'Policy N°', 
    how= 'outer', 
    suffixes= ('_AE','_VA')
)
# Second merge, previous with CL
df_payments_month = pd.merge(
    df_payments_month,
    df_CL_final,
    on = 'Policy N°',
    how= 'outer'
)

# Data format
# Rename to normalize the policy number column
df_payments_month.rename(columns = {'Policy N°': 'Póliza','Monto_AE': 'Expenses_Paid_conta', 'Monto_VA': 'Valuation_Expenses_conta','Monto':'Claims_Paid_conta' }
                         ,inplace = True)
# Debug
df_payments_month.to_excel(f'{ruta_procesados}/pruebas/df_payments_month.xlsx', index= False)

In [90]:
# Final merge of validation 2

# First merge, pivot and monthly databases
df_val_2 = pd.merge(df_payments_piv,
    df_payments_base,
    on = 'Póliza',
    how= 'outer'
)

# Second merge, result and monthly payments
df_val_2 = pd.merge(df_val_2,
    df_payments_month,
    on = 'Póliza',
    how= 'outer'
)

# NaN handling
df_val_2.fillna(0, inplace=True)

# Creation of validation columns
df_val_2['Expenses_Paid_Validacion'] = df_val_2[f'Expenses_Paid_{AñoMes_piv}'] + df_val_2[f'Expenses_Paid_conta'] - df_val_2[f'Expenses_Paid_{AñoMes}']  
# Creation of validation columns
df_val_2['Valuation_Expenses_Validacion'] = df_val_2[f'Valuation_Expenses_{AñoMes_piv}'] + df_val_2[f'Valuation_Expenses_conta'] - df_val_2[f'Valuation_Expenses_{AñoMes}'] 
# Creation of validation columns
df_val_2['Claims_Paid_Validacion'] = df_val_2[f'Claims_Paid_{AñoMes_piv}'] + df_val_2[f'Claims_Paid_conta'] - df_val_2[f'Claims_Paid_{AñoMes}'] 


## 7. Final Export

In [91]:
# =============================================================================
# Section 7: Final Export
# =============================================================================

# Name of xlsx file
output_final = f'{route_validations}/{AñoMes}_Validaciones_Contables_Marine.xlsx'

# Use of ExcelWriter to write the final xlsx file
with pd.ExcelWriter(output_final, engine='xlsxwriter') as writer:
    # Export df_val_1 to sheet 'Validacion OSLR'
    df_val_1.to_excel(writer, sheet_name='Validacion OSLR', index=False)
    # Export df_val_2 to sheet 'Validacion PAGOS'
    df_val_2.to_excel(writer, sheet_name='Validacion PAGOS', index=False)


print(f'Se exportó el archivo correctamente como {output_final}')
print('===============================================================')
print(f'VALIDACION CONTABLE DE PAGOS AL MES {AñoMes} realizada correctamente')
print('===============================================================')

Se exportó el archivo correctamente como C:/Users/IKAL14/Documents/Integral/Marine/Validaciones/202604/202604_Validaciones_Contables_Marine.xlsx
VALIDACION CONTABLE DE PAGOS AL MES 202604 realizada correctamente


In [92]:
end_time = timeit.default_timer()
print(f"El programa terminó de correr correctamente en: {round(end_time - start_time,2)} s")

El programa terminó de correr correctamente en: 1.01 s
